<a href="https://colab.research.google.com/github/roshansinghal001-droid/blinkit-sales-dashboard/blob/main/Weather_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Weather Data Analysis
### Analyzing historical hourly weather data using Pandas

In [4]:
## 1. Introduction

'''Weather is something that affects daily life, agriculture, travel, and planning, yet raw
weather sensor data is often messy — missing values, inconsistent formats, and data spread
across multiple files. This project analyzes historical hourly weather data collected from
multiple cities between 2012–2017, with the goal of cleaning, organizing, and exploring the
data to uncover temperature trends, seasonal patterns, and extreme weather days.

**Dataset:** [Historical Hourly Weather Data](https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data)
(Kaggle) — hourly readings of temperature, humidity, pressure, wind speed, wind direction,
and weather description across 30+ North American cities.

**Objective:**
- Clean and merge multiple raw weather datasets into a single, analysis-ready dataset
- Handle missing/inconsistent sensor data using appropriate strategies (interpolation vs.
  dropping unreliable columns)
- Identify hottest and coldest recorded days
- Analyze seasonal and city-wise temperature trends

**Tools used:** Python, Pandas'
'''

"Weather is something that affects daily life, agriculture, travel, and planning, yet raw \nweather sensor data is often messy — missing values, inconsistent formats, and data spread \nacross multiple files. This project analyzes historical hourly weather data collected from \nmultiple cities between 2012–2017, with the goal of cleaning, organizing, and exploring the \ndata to uncover temperature trends, seasonal patterns, and extreme weather days.\n\n**Dataset:** [Historical Hourly Weather Data](https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data) \n(Kaggle) — hourly readings of temperature, humidity, pressure, wind speed, wind direction, \nand weather description across 30+ North American cities.\n\n**Objective:**\n- Clean and merge multiple raw weather datasets into a single, analysis-ready dataset\n- Handle missing/inconsistent sensor data using appropriate strategies (interpolation vs. \n  dropping unreliable columns)\n- Identify hottest and coldest recorded

In [ ]:
## 2. Loading the Data

In [5]:
import pandas as pd
ca = pd.read_csv('/content/city_attributes.csv')
wind_des = pd.read_csv('/content/weather_description.csv')
temp = pd.read_csv('/content/temperature.csv')
wd= pd.read_csv('/content/wind_direction.csv')
ws = pd.read_csv('/content/wind_speed.csv')
hum= pd.read_csv('/content/humidity.csv')
pre = pd.read_csv('/content/pressure.csv')

In [ ]:
## 3. Data Cleaning

In [6]:
# changing data type of table 'WIND DESCRIPTION'

wind_des['datetime']=pd.to_datetime(wind_des['datetime'])
print(wind_des['datetime'].dtype)

# changing data type of table 'TEMPERATURE'

temp['datetime']=pd.to_datetime(temp['datetime'])
print(temp['datetime'].dtype)


# changing data type of table 'WIND DIRECTION'

wd['datetime'] = pd.to_datetime(wd['datetime'])
print(wd['datetime'].dtype)


# changing data type of table 'WIND SPEED'

ws['datetime']=pd.to_datetime(ws['datetime'])
print(ws['datetime'].dtype)


#changing data type of table 'HUMIDITY'

hum['datetime']=pd.to_datetime(hum['datetime'])
print(hum['datetime'].dtype)


#changing data type of table 'PRESSURE'
pre['datetime']=pd.to_datetime(pre['datetime'])
print(pre['datetime'].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


In [7]:
# Finding and filling the null values

# 'WIND DESCRIPTION'

city_cols = wind_des.columns.drop('datetime')

wind_des[city_cols] = wind_des[city_cols].ffill()
wind_des[city_cols] = wind_des[city_cols].bfill()

wd.isna().sum().sum()   # confirm it's 0



# 'TEMPRETURE'

temp.isna().sum().sort_values(ascending=True)

city_cols = temp.columns.drop('datetime')

temp[city_cols] = temp[city_cols].interpolate()
temp[city_cols] = temp[city_cols].bfill()

temp.isna().sum().sum()   # confirm it's 0

# CONVERTING KELVIN TO CELUIS
temp[city_cols] = temp[city_cols] - 273.15



# 'WIND DIRECTION'

wd.isna().sum().sort_values(ascending=True)

city_cols= wd.columns.drop('datetime')

wd[city_cols]=wd[city_cols].interpolate()
wd[city_cols]=wd[city_cols].bfill()

wd.isna().sum().sum()       # confirm it's 0




# 'WIND SPEED'

ws.isna().sum().sort_values(ascending=True)

city_cols= ws.columns.drop('datetime')

ws[city_cols]=ws[city_cols].interpolate()
ws[city_cols]=ws[city_cols].bfill()

ws.isna().sum().sum()         # confirm it's 0



#'HUMIDITY'

hum.isna().sum().sort_values(ascending=True)

''' CONVERTING MISSING VALUES TO PERCENTAGE'''
missing_per = (hum.isna().sum()/len(hum))*100

missing_per.sort_values(ascending=True)



city_cols = hum.columns.drop('datetime')

hum[city_cols] = hum[city_cols].interpolate()
hum[city_cols]=hum[city_cols].bfill()

hum.isna().sum().sum()        # confirm it's 0




# 'PRESSURE '


pre.isna().sum().sort_values(ascending=True)

# drop Montreal and Vancouver (too much missing data)
pressure = pre.drop(columns=['Montreal', 'Vancouver'])

# interpolate + backfill the rest
city_cols = pre.columns.drop('datetime')
pre[city_cols] = pre[city_cols].interpolate()
pre[city_cols] = pre[city_cols].bfill()

pre.isna().sum().sum()   # confirm 0

np.int64(0)

In [ ]:
## 4. Feature Engineering (date/season columns)

In [8]:
'''
Melting all the tables
'''

temp_long = temp.melt(id_vars='datetime',var_name='city',value_name='temperature')
humidity_long = hum.melt(id_vars='datetime', var_name='city', value_name='humidity')
pressure_long = pre.melt(id_vars='datetime', var_name='city', value_name='pressure')
wind_speed_long = ws.melt(id_vars='datetime', var_name='city', value_name='wind_speed')
wind_direction_long = wd.melt(id_vars='datetime', var_name='city', value_name='wind_direction')
weather_description_long = wd.melt(id_vars='datetime', var_name='city', value_name='weather_description')

In [9]:
'''
MERGING ALL THE TABLE INTO ONE MASTER TABLE
'''


weather = temp_long.merge(humidity_long, on=['datetime', 'city'], how='inner')
weather = weather.merge(pressure_long, on=['datetime', 'city'], how='inner')
weather = weather.merge(wind_speed_long, on=['datetime', 'city'], how='inner')
weather = weather.merge(wind_direction_long, on=['datetime', 'city'], how='inner')
weather = weather.merge(weather_description_long, on=['datetime', 'city'], how='inner')




''' CHECKING NULL VALUES IN MASTER TABLE '''

weather.isna().sum()

,0
datetime,0
city,0
temperature,0
humidity,0
pressure,0
wind_speed,0
wind_direction,0
weather_description,0


In [ ]:
## 5. Analysis & Key Findings

In [10]:
'''  DEFINING THE TIMEZONE '''

weather['year']= weather['datetime'].dt.year
weather['month']= weather['datetime'].dt.month
weather['day']= weather['datetime'].dt.day
weather['hour']= weather['datetime'].dt.hour


''' DEFINING THE SEASON '''
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

weather['season'] = weather['month'].apply(get_season)






# Hottest day overall
hottest=weather.loc[weather['temperature'].idxmax()]

# Coldest day overall
coldest=weather.loc[weather['temperature'].idxmin()]

# Average temperature by season
seasonal_avg=weather.groupby('season')['temperature'].mean().sort_values(ascending=False)

# Average temperature by city
city_avg=weather.groupby('city')['temperature'].mean().sort_values(ascending=False)

In [11]:
## 6. Conclusion

'''
This project cleaned and merged 6 real-world weather datasets into one analysis-ready
dataset (~130K records). Key challenges included handling inconsistent missing data
across cities and reshaping wide-format sensor data into tidy format for analysis.
Findings show clear seasonal temperature patterns and city-level climate differences.
'''

'\nThis project cleaned and merged 6 real-world weather datasets into one analysis-ready \ndataset (~130K records). Key challenges included handling inconsistent missing data \nacross cities and reshaping wide-format sensor data into tidy format for analysis. \nFindings show clear seasonal temperature patterns and city-level climate differences.\n'